In [3]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm


import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson

CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


In [4]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41,42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [5]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2):
    data_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/explanation_data"
    file_path = os.path.join(data_dir, f"subject_{subject_index}_results.pkl")

    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f)   
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [6]:
def load_subject_concept(subject_index=2, freq_band='alpha'):
    #pd.read_csv("connectivity_subject_{subject_index}_delta.csv")
    # Load connectivity data for the specified subject
    file_path = f"connectivity_subject_{subject_index}_{freq_band}.csv"
    
    # Try to load the subject-specific file, fall back to subject 80 if not found

    connectivity_data = pd.read_csv(file_path)
    
    # Filter out trials that should be ignored (first 150 trials)
    filtered_data = connectivity_data[connectivity_data['trial_index'] >= 150]
    filtered_data = filtered_data.drop(columns=['subject_index', 'ch_index1', 'ch_index2'])
    
    # Extract relevant information: trial index, channel combinations, and concept values

    return filtered_data

extract trial index, channel name combination and concept of choice. compute correlation with predited amplitude
member to igore first 150 trials

In [32]:
predictions, uncertainties, explanations, ch_names = load_predicted_amplitude_for_subject(subject_index=80)

Loading EEG data...

subject index: 80, frequency band: None

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_080_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_080_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
760 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


In [33]:
df =load_subject_concept(subject_index=80, freq_band='delta')

In [34]:
df

,trial_index,ch_name1,ch_name2,icoh,pli,wpli
265500,150,Fp1,Fp2,0.215285,0.453333,0.916487
265501,150,Fp1,F3,0.027583,0.048889,0.051001
265502,150,Fp1,F4,0.045193,0.113333,0.062339
265503,150,Fp1,C3,0.042632,0.148889,0.088905
265504,150,Fp1,C4,0.524520,0.604444,0.710856
...,...,...,...,...,...,...
1345195,759,Fpz,POz,0.052498,0.360000,0.139311
1345196,759,Fpz,Oz,0.289568,0.673333,0.698502
1345197,759,CPz,POz,0.125770,0.266667,0.248588
1345198,759,CPz,Oz,0.255873,0.424444,0.463601


In [35]:
trial_indices = sorted(df['trial_index'].unique())
prediction_values = [predictions[i-151] for i in trial_indices] 

In [36]:
len(prediction_values)

610

In [37]:
roi_channels = ['C1', 'C3', 'C4', 'C5',  'FC1', 'FC3', 'FC4', 'FC5', 'Fz', 'F1', 'F3', 'CPz', 'Cz', 'CP1', 'CP5', 'CP3', 'CP4' ]

In [38]:
import itertools
roi_channel_pairs = list(itertools.combinations(roi_channels, 2))

In [1]:
from scipy.stats import pearsonr
def plot_channel_pair_correlations(df, predictions, roi_channel_pairs, concept='icoh', n_cols=4, show_plots=False):
    """
    Plot correlations between predictions and a given concept for ROI channel pairs.
    Only shows plots with absolute correlation > 0.2 and displays each plot individually.
    
    Args:
        df: DataFrame containing the connectivity data
        predictions: Array of prediction values
        roi_channel_pairs: List of channel pairs to analyze
        concept: Column name in df to analyze (default='icoh')
        n_cols: Number of columns in the subplot grid (default=4)
    """


    results = {}

    # For each ROI channel pair, create an individual plot if correlation is significant
    for ch1, ch2 in roi_channel_pairs:
        pairs = []
        pairs.append((ch1, ch2))
        pairs.append((ch2, ch1))  # Add reversed pairs too
        valid_pairs = pd.DataFrame(pairs, columns=['ch_name1', 'ch_name2'])
        # Use merge to efficiently filter for matching channel pairs
        filtered_df = df.merge(valid_pairs, on=['ch_name1', 'ch_name2'], how='inner')
        # Filter data for this channel pair (check both directions)
        pair_data = filtered_df[((filtered_df['ch_name1'] == ch1) & (filtered_df['ch_name2'] == ch2)) | 
                              ((filtered_df['ch_name1'] == ch2) & (filtered_df['ch_name2'] == ch1))]
        

        #pair_trials = pair_data['trial_index'].values
        concept_values = pair_data[concept].values
            
        # Create trial_index to concept value mappin
        if len(concept_values) == 0:
            continue        
        #corr = np.corrcoef(np.array(predictions), np.array(concept_values))[0, 1]
        corr_object = pearsonr(np.array(concept_values), predictions)
        #all_results[(ch1, ch2)] = corr_object
        corr = corr_object[0]
        pval = corr_object[1]
        if pval< (0.05/60):
            results[(ch1, ch2)] = {"corr": corr, "pval": pval}

        
        if show_plots: # Only show plots with absolute correlation > 0.2
            if abs(corr) > 0.2:
                plt.figure(figsize=(6, 4))
                
            # Create scatter plot
                plt.scatter(np.array(concept_values), np.array(predictions), alpha=0.6)
                plt.title(f'{ch1}-{ch2} Correlation with {concept.upper()} (r={corr:.3f}) {band_name}')
                plt.ylabel('Predicted Amplitude')
                plt.xlabel(concept.upper())
                #results[(ch1, ch2)] = corr

    return results


In [9]:
mne.set_log_level("ERROR")

In [10]:
import itertools
import os
import numpy as np
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    roi_channel_pairs = list(itertools.combinations(ch_names, 2))
    print(f"Subject {subject_index}")
    predictions, uncertainties, explanations, ch_names = load_predicted_amplitude_for_subject(subject_index=subject_index)
    for concept in ['icoh', 'wpli', 'pli']:
        save_dir = f"results/{concept}"
        print(concept)
        result = {}
        for band_name in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
            print(band_name)
            df = load_subject_concept(subject_index=subject_index, freq_band=
            band_name)
    
        
            result[band_name] = plot_channel_pair_correlations(df, predictions, roi_channel_pairs, concept=concept, n_cols=4)
        os.makedirs(save_dir, exist_ok=True)
        file_name = f"subject_{subject_index}_{concept}.npy"
        np.save(os.path.join(save_dir, file_name), result)



Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
Subject 1
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
icoh
delta
theta
alpha
beta
gamma
wpli
delta
theta
alpha
beta
gamma
pli
delta
theta
alpha
beta
gamma
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
Subject 2
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
icoh
delta
theta
alpha
beta
gamma
wpli
delta
theta
alpha
beta
gamma
pli
delta
theta
alpha
beta
gamma
Loading EEG data...

subject index: 13, frequency band: None

(100, 60, 900)
(1, 60, 1)
Subject 13
Loading EEG data...

subject index: 13, frequency band: None

(100, 60, 900)
(1, 60, 1)
icoh
delta
theta
alpha
beta
gamma
wpli
delta
theta
alpha
beta
gamma
pli
delta
theta
alpha
beta
gamma
Loading EEG data...

subject index: 24, frequency band: None

(100, 60, 900)
(1, 60, 1)
Subject 24
Loading EEG data...

subject index: 24, fre

FileNotFoundError: [Errno 2] No such file or directory: 'connectivity_subject_92_delta.csv'